# Staged Calibration: Simulation vs. Experiment

**Branch:** `paper/results`  
**Solver:** legacy `relax_to_steady`, no distributed leakage, no BE-CV  
**Tests:** 0116, 0121 (17 ft) | 0123, 0126 (15 ft)  

Stages:
- **A** Momentum / resistance (closed-door): A_leak, Cd, K_stove, K_minor_mult
- **B** Heat-loss (closed-door): h_inner_mult, h_outer_mult, eps_inner_scale
- **C** Cap Cp (optional)
- **D** Cracked-door refinement (optional)

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import time as _time
import datetime
import json
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'dev' / 'exp_data'))

from flued_module.chimney import Chimney
from flued_module.material_properties import MaterialProperties
from experiment_loader import ExperimentLoader
from flued_module.calibration.param_space import (
    StageAParams, StageBParams, StageCParams, StageDParams,
)
from flued_module.calibration.objectives import (
    compute_stage_A_objective, compute_stage_B_objective,
    compute_stage_C_objective,
)
from flued_module.calibration.calibrator import StagedCalibrator

matplotlib.rcParams['figure.dpi'] = 120

mat_props = MaterialProperties(str(REPO_ROOT / 'flued_module' / 'data' / 'pipe_props.json'))
el = ExperimentLoader(
    metadata_path=REPO_ROOT / 'dev' / 'exp_data' / 'experiments_metadata.json'
)
print('Setup complete')

## Configuration

In [ ]:
# ============================================================
#  Tests
# ============================================================
TEST_IDS      = ['0116', '0121', '0123', '0126']
REP_TEST      = '0116'   # representative for plots

# ============================================================
#  Fixed physical parameters (NOT calibrated)
# ============================================================
P_ATM          = 98457
TC1_OFFSET_C   = 10.0
DOOR_WIDTH_IN  = 15.25
DOOR_HEIGHT_IN =  9.875
DOOR_RAMP_S    =  5.0
DT             =  1.0
SIM_TIME_MAX   = 1200

DIAMETER_IN    = 6.0
DIAMETER_FT    = DIAMETER_IN / 12
DIAMETER_M     = DIAMETER_IN * 0.0254
PIPE_AREA_M2   = np.pi * (DIAMETER_M / 2)**2

FINE_TOP_IN    = 15
FINE_TOP_SEGS  = 15
BLACK_PIPE_IN  = 73.5
TEE1_VERT_IN   = 12.0;  TEE1_HORIZ_IN  = 6.5
ELBOW_VERT_IN  =  7.5;  ELBOW_HORIZ_IN = 8.5
HORIZ_IN       = 23.25
TEE2_VERT_IN   = 12.0;  TEE2_HORIZ_IN  = 6.5

PRESSURE_SUBSAMPLE = 10

# ============================================================
#  Sensor maps (same as results_validation)
# ============================================================
SENSOR_MAP = {
    '15ft': {'TC1': (0,15.0), 'TC2': (2,12.0), 'TC3': (2,42.0), 'TOP': (3,3.0)},
    '17ft': {'TC1': (0,15.0), 'TC6': (2,3.0), 'TC2': (2,34.75), 'TC3': (2,64.75), 'TOP': (3,3.0)},
}
TEMP_SENSORS  = {'15ft': ['TC1','TC2','TC3'], '17ft': ['TC1','TC6','TC2','TC3']}
PRESS_SENSORS = {'15ft': ['TC1','TC2','TC3','TOP'], '17ft': ['TC1','TC6','TC2','TC3','TOP']}

# ============================================================
#  Sweep sizes (TUNE THESE for smoke test vs full run)
# ============================================================
N_COARSE_A = 5     # Stage A coarse samples (set 40-60 for full run)
N_COARSE_B = 5     # Stage B coarse samples (set 20-40 for full run)
N_REFINE_TOP = 3   # number of top candidates to refine
REFINE_MAXITER = 20 # local optimizer iterations per candidate
RNG_SEED = 42

# ============================================================
#  Objective weights
# ============================================================
W_MDOT  = 1.0      # mdot normalized by sigma_mdot
W_PRESS = 0.25     # pressure (lower weight; noisier, more sensors)
SIGMA_P = 2.0      # Pa -- conservative pressure noise scale

# ============================================================
#  Baseline parameters (paper/results defaults)
# ============================================================
BASELINE_A = StageAParams(A_leak_cm2=75.0, Cd=0.62, K_stove=0.0, K_minor_mult=1.0)
BASELINE_B = StageBParams(h_inner_mult=1.0, h_outer_mult=1.0, eps_inner_scale=1.0)

# ============================================================
#  Output directory
# ============================================================
TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT  = REPO_ROOT / 'outputs' / 'calibration' / TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Output -> {OUT_ROOT}')
print(f'Coarse samples: A={N_COARSE_A}, B={N_COARSE_B}')
print(f'Baseline A: {BASELINE_A.to_dict()}')
print(f'Baseline B: {BASELINE_B.to_dict()}')

## Helpers (reused from results_validation)

In [ ]:
def smooth_transition(t, t_start, t_end, val_start, val_end):
    if t <= t_start: return val_start
    if t >= t_end:   return val_end
    frac = (t - t_start) / (t_end - t_start)
    return val_start + (val_end - val_start) * 0.5 * (1.0 - np.cos(np.pi * frac))


def build_parts(chimney_type, T_indoor_K, T_outdoor_K):
    geom = el.get_chimney_geometry(chimney_type)
    bulk_vert_in = geom['final_vert_in'] - FINE_TOP_IN
    return [
        mat_props.create_single_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=BLACK_PIPE_IN/12,
            z_ft=0, material='steel_24_gauge', orientation='vertical',
            T_ambient_K=T_indoor_K, Pa=P_ATM, num_segments=6),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=HORIZ_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='horizontal',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=2),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=bulk_vert_in/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=8),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=FINE_TOP_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=FINE_TOP_SEGS),
    ]


def build_fittings(T_indoor_K, T_outdoor_K, vane_factor=1.0):
    """Build fittings list. vane_factor multiplies all K-factors."""
    return [
        {'type': 'tee', 'between_parts': (-1, 0),
         'horizontal_length_ft': TEE1_HORIZ_IN/12,
         'vertical_length_ft':   TEE1_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K,
         'vane_factor': vane_factor},
        {'type': 'elbow', 'between_parts': (0, 1),
         'height_ft': ELBOW_VERT_IN/12, 'width_ft': ELBOW_HORIZ_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K,
         'angle_deg': 90, 'R_over_D': 1.5, 'elbow_type': 'long_radius',
         'vane_factor': vane_factor},
        {'type': 'tee', 'between_parts': (1, 2),
         'horizontal_length_ft': TEE2_HORIZ_IN/12,
         'vertical_length_ft':   TEE2_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'triple',
         'inner_material': 'stainless_430', 'middle_material': 'ceramic_fiber',
         'outer_material': 'stainless_316', 'T_ambient_K': T_outdoor_K,
         'vane_factor': vane_factor},
    ]


def make_door_func(test_id, ramp_s=None):
    cfg     = el.get_test_config(test_id)
    offsets = cfg['door_events']['offsets_s']
    door_in = cfg['door_events'].get('door_opening_in', 2.0)
    RAMP    = ramp_s if ramp_s is not None else DOOR_RAMP_S
    cracked_pct = (
        np.degrees(np.arcsin(
            min(door_in, DOOR_HEIGHT_IN * 0.999) / DOOR_HEIGHT_IN
        )) / 90.0 * 100.0
    )
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_reopen = offsets.get('door_reopen_2in', None)
    t_close2 = offsets.get('door_closed_again', None)

    def door_func(t):
        if t < t_crack - RAMP:
            return 100.0
        if t < t_crack:
            return smooth_transition(t, t_crack - RAMP, t_crack, 100.0, cracked_pct)
        if t < t_close:
            return cracked_pct
        if t < t_close + RAMP:
            return smooth_transition(t, t_close, t_close + RAMP, cracked_pct, 0.0)
        if t_reopen is not None:
            if t < t_reopen:
                return 0.0
            if t < t_reopen + RAMP:
                return smooth_transition(t, t_reopen, t_reopen + RAMP, 0.0, cracked_pct)
            if t_close2 is not None and t < t_close2:
                return cracked_pct
            if t_close2 is not None and t < t_close2 + RAMP:
                return smooth_transition(t, t_close2, t_close2 + RAMP, cracked_pct, 0.0)
        return 0.0
    return door_func


def find_segment(ch, part_idx, height_in):
    target_m = ch.part_base_elevations[part_idx] + height_in * 0.0254
    return int(np.argmin(np.abs(ch.segment_elevations - target_m)))


def get_phase_windows(test_id):
    """Return (t_cracked_start, t_cracked_end, t_closed_start, t_closed_end)."""
    offsets  = el.get_test_config(test_id)['door_events']['offsets_s']
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_end    = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)
    return (t_crack, t_close, t_close + 30, t_end)


def compute_rmse_bias(exp_t, exp_v, sim_t, sim_v, t_window):
    exp_t = np.asarray(exp_t, dtype=float)
    exp_v = np.asarray(exp_v, dtype=float)
    mask  = (exp_t >= t_window[0]) & (exp_t <= t_window[1])
    et, ev = exp_t[mask], exp_v[mask]
    if len(et) < 5:
        return np.nan, np.nan
    sv   = np.interp(et, sim_t, sim_v)
    diff = sv - ev
    return np.sqrt(np.nanmean(diff**2)), float(np.nanmean(diff))


def savefig(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  saved: {path}')


def get_pitot_mdot(test_id):
    dp_data  = el.get_sensor_data(test_id, 'pitot_dp', as_gauge=False)
    tc4_data = el.get_temperature_data(test_id, 'TC4')
    top_data = el.get_sensor_data(test_id, 'TOP', as_gauge=True)
    t_dp = np.asarray(dp_data['time_s'],      dtype=float)
    dp   = np.asarray(dp_data['pressure_Pa'],  dtype=float)
    t_tc4 = np.asarray(tc4_data['time_s'],        dtype=float)
    T_tc4 = np.asarray(tc4_data['temperature_C'], dtype=float) + 273.15
    t_top = np.asarray(top_data['time_s'],      dtype=float)
    P_top = np.asarray(top_data['pressure_Pa'], dtype=float)
    T_g = np.interp(t_dp, t_tc4, T_tc4)
    P_g = np.interp(t_dp, t_top, P_top)
    R_AIR = 287.0
    rho   = (P_ATM + P_g) / (R_AIR * np.maximum(T_g, 200.0))
    v     = np.sqrt(2.0 * np.maximum(dp, 0.0) / np.maximum(rho, 0.1))
    mdot  = rho * v * PIPE_AREA_M2
    dt_est = max(float(np.median(np.diff(t_dp))), 0.1)
    win    = max(1, int(10.0 / dt_est))
    mdot_sm = uniform_filter1d(mdot, size=win)
    return t_dp, mdot * 1e3, mdot_sm * 1e3


print('Helpers defined.')

## Load experimental data

In [ ]:
exp_data = {}
for test_id in TEST_IDS:
    cfg = el.get_test_config(test_id)
    ct  = cfg['chimney_type']
    temp  = {s: el.get_temperature_data(test_id, s) for s in TEMP_SENSORS[ct]}
    press = {s: el.get_sensor_data(test_id, s, as_gauge=True)
             for s in PRESS_SENSORS[ct]}
    exp_data[test_id] = {
        'chimney_type': ct, 'config': cfg,
        'temperature': temp, 'pressure': press,
    }
    cond = cfg['conditions']
    print(f'{test_id} ({ct}): wind={cond["wind_speed_m_s"]:.1f} m/s, '
          f'T_in={cond["T_indoor_C"]:.0f} C, T_out={cond["T_outdoor_C"]:.0f} C')

# Pitot calibration JSON for exp SS mdot
_pj = REPO_ROOT / 'dev' / 'notebooks' / 'validation' / 'pitot_calibration_results.json'
with open(_pj) as f:
    pitot_cal = json.load(f)

# Pre-compute experimental SS pressure at each sensor (closed-door window)
exp_ss_press = {}  # test_id -> {sensor: mean_Pa}
exp_ss_mdot = {}
exp_ss_mdot_std = {}

for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    _, _, t_cls0, t_cls1 = get_phase_windows(tid)
    exp_ss_press[tid] = {}
    for s in PRESS_SENSORS[ct]:
        pd_ = exp_data[tid]['pressure'][s]
        et = np.asarray(pd_['time_s'], dtype=float)
        ev = np.asarray(pd_['pressure_Pa'], dtype=float)
        mask = (et >= t_cls0) & (et <= t_cls1)
        exp_ss_press[tid][s] = float(np.mean(ev[mask])) if mask.sum() > 0 else np.nan

    if tid in pitot_cal:
        exp_ss_mdot[tid]     = pitot_cal[tid]['experimental']['mdot_kgs'] * 1000
        exp_ss_mdot_std[tid] = pitot_cal[tid]['experimental']['mdot_std_kgs'] * 1000
    else:
        exp_ss_mdot[tid]     = np.nan
        exp_ss_mdot_std[tid] = np.nan

print(f'\nLoaded {len(exp_data)} tests.')
print('Exp SS mdot (g/s):', {k: f'{v:.1f}' for k, v in exp_ss_mdot.items()})

## Simulation runner with calibration knobs

In [ ]:
def run_sim_calibrated(test_id, stage_a=None, stage_b=None, stage_c=None,
                       stage_d=None):
    """Run one simulation with calibration knobs applied.

    Parameters
    ----------
    test_id : str
    stage_a : StageAParams or dict (or None for baseline)
    stage_b : StageBParams or dict (or None for baseline)
    stage_c : StageCParams or dict (or None for baseline)
    stage_d : StageDParams or dict (or None for baseline)

    Returns
    -------
    ch : Chimney object after run_simulation()
    """
    # Unpack stage params
    if stage_a is None:
        stage_a = BASELINE_A.to_dict()
    elif hasattr(stage_a, 'to_dict'):
        stage_a = stage_a.to_dict()

    if stage_b is None:
        stage_b = BASELINE_B.to_dict()
    elif hasattr(stage_b, 'to_dict'):
        stage_b = stage_b.to_dict()

    if stage_c is not None and hasattr(stage_c, 'to_dict'):
        stage_c = stage_c.to_dict()

    if stage_d is not None and hasattr(stage_d, 'to_dict'):
        stage_d = stage_d.to_dict()

    # Test config
    cfg      = el.get_test_config(test_id)
    ct       = cfg['chimney_type']
    cond     = cfg['conditions']
    offsets  = cfg['door_events']['offsets_s']
    T_in_K   = cond['T_indoor_C']  + 273.15
    T_out_K  = cond['T_outdoor_C'] + 273.15
    wind_ms  = cond['wind_speed_m_s']

    # Stage A: inlet restriction + fittings multiplier
    A_leak_m2    = stage_a['A_leak_cm2'] * 1e-4
    Cd           = stage_a['Cd']
    K_stove      = stage_a['K_stove']
    K_minor_mult = stage_a['K_minor_mult']

    parts    = build_parts(ct, T_in_K, T_out_K)
    fittings = build_fittings(T_in_K, T_out_K, vane_factor=K_minor_mult)

    # Stage D: door ramp
    ramp_s = stage_d.get('door_ramp_s', DOOR_RAMP_S) if stage_d else DOOR_RAMP_S
    door_fn = make_door_func(test_id, ramp_s=ramp_s)

    # Inlet temperature from TC1 + offset
    tc1_data = el.get_temperature_data(test_id, 'TC1')
    tc1_K    = np.asarray(tc1_data['temperature_C'], dtype=float) + TC1_OFFSET_C + 273.15
    inlet_fn = interp1d(tc1_data['time_s'], tc1_K,
                        kind='linear', bounds_error=False, fill_value='extrapolate')

    sim_time = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)

    inlet_cfg = {
        'enable': True, 'door_type': 'swivel',
        'door_width_ft':   DOOR_WIDTH_IN / 12,
        'door_height_ft':  DOOR_HEIGHT_IN / 12,
        'Cd': Cd,
        'A_leak_m2': A_leak_m2,
        'inlet_area_min_m2': 0.0,
        'K_stove': K_stove,
    }

    # Stage C: cap Cp override
    top_bnd = {
        'enable_wind_cap_effect': True,
        'cap_type': 'rain_cap',
        'wind_speed_m_s': wind_ms,
    }
    if stage_c and 'Cp_override' in stage_c:
        top_bnd['Cp_override'] = stage_c['Cp_override']

    ch = Chimney(
        parts=parts, elbows=fittings,
        dt=DT, total_time=sim_time,
        use_coupled_solver=True,
        door_opening_func=door_fn,
        inlet_temp_func=inlet_fn,
        inlet_restriction=inlet_cfg,
        top_boundary=top_bnd,
        leakage_config=None,      # EXPLICIT: no distributed leakage
        enable_diagnostics=False,  # speed: skip diagnostics during calibration
        verbose=False,
    )

    # Stage B: heat transfer multipliers (set AFTER construction, BEFORE run)
    if hasattr(ch, 'comprehensive_ht'):
        ch.comprehensive_ht.h_inner_multiplier = stage_b.get('h_inner_mult', 1.0)
        ch.comprehensive_ht.h_outer_multiplier = stage_b.get('h_outer_mult', 1.0)
        # eps_inner_scale: modify liner emissivity on all triple-wall parts
        eps_scale = stage_b.get('eps_inner_scale', 1.0)
        if eps_scale != 1.0:
            for part in ch.parts:
                if hasattr(part, 'wall_properties'):
                    wp = part.wall_properties
                    if hasattr(wp, 'liner') and hasattr(wp.liner, 'emissivity'):
                        wp.liner.emissivity = min(1.0, wp.liner.emissivity * eps_scale)
                    elif hasattr(wp, 'emissivity'):
                        wp.emissivity = min(1.0, wp.emissivity * eps_scale)

    ch.run_simulation()
    return ch


print('run_sim_calibrated defined.')

## Result extraction

In [ ]:
def extract_results(test_id, ch):
    """Extract SS mdot, SS pressure, and temperature RMSE from a completed sim."""
    ct = exp_data[test_id]['chimney_type']
    _, _, t_cls0, t_cls1 = get_phase_windows(test_id)
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt
    n = ch.T_gas.shape[0]

    # SS mdot (closed-door)
    mask_ss = (t_sim >= t_cls0) & (t_sim <= t_cls1)
    mdot = ch.mass_flow_rate
    if mdot.ndim > 1:
        mdot = mdot[:, 0]
    mdot_ss = float(np.mean(mdot[mask_ss])) * 1000 if mask_ss.sum() > 0 else np.nan

    # SS pressure at each sensor (closed-door)
    seg_map = {s: find_segment(ch, *SENSOR_MAP[ct][s]) for s in PRESS_SENSORS[ct]}
    step_indices = np.arange(0, n, PRESSURE_SUBSAMPLE)
    p_times = step_indices * ch.dt
    unique_segs = list(set(seg_map.values()))

    seg_P = {seg: np.zeros(len(step_indices)) for seg in unique_segs}
    for i, step in enumerate(step_indices):
        prof = ch.get_static_pressure_profile(timestep=int(step))
        for seg in unique_segs:
            seg_P[seg][i] = prof['P_gauge_Pa'][seg]

    press_ss = {}
    p_mask = (p_times >= t_cls0) & (p_times <= t_cls1)
    for s in PRESS_SENSORS[ct]:
        vals = seg_P[seg_map[s]][p_mask]
        press_ss[s] = float(np.mean(vals)) if len(vals) > 0 else np.nan

    # Temperature RMSE per sensor (closed-door)
    temp_rmse = {}
    for sensor in TEMP_SENSORS[ct]:
        seg = find_segment(ch, *SENSOR_MAP[ct][sensor])
        sim_T = ch.T_gas[:, seg] - 273.15
        td = exp_data[test_id]['temperature'][sensor]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)
        rmse, _ = compute_rmse_bias(et, ev, t_sim, sim_T, (t_cls0, t_cls1))
        temp_rmse[sensor] = rmse

    return {
        'mdot_ss': mdot_ss,
        'press_ss': press_ss,
        'temp_rmse': temp_rmse,
    }


print('extract_results defined.')

## Evaluation functions for calibrator

In [ ]:
# These closures capture the frozen params from earlier stages.
# They are redefined when a stage is completed.

_frozen_a = BASELINE_A.to_dict()
_frozen_b = BASELINE_B.to_dict()
_frozen_c = None


def eval_stage_A(params_dict):
    """Run all tests with Stage A params, return (J_A, details)."""
    ss_mdot_sim = {}
    ss_press_sim = {}
    for tid in TEST_IDS:
        ch = run_sim_calibrated(tid, stage_a=params_dict)
        res = extract_results(tid, ch)
        ss_mdot_sim[tid] = res['mdot_ss']
        ss_press_sim[tid] = res['press_ss']

    press_sensors_per_test = {tid: PRESS_SENSORS[exp_data[tid]['chimney_type']]
                              for tid in TEST_IDS}

    J, details = compute_stage_A_objective(
        ss_mdot_sim=ss_mdot_sim,
        ss_mdot_exp=exp_ss_mdot,
        ss_mdot_exp_std=exp_ss_mdot_std,
        ss_pressure_sim=ss_press_sim,
        ss_pressure_exp=exp_ss_press,
        test_ids=TEST_IDS,
        press_sensors_per_test=press_sensors_per_test,
        w_mdot=W_MDOT, w_press=W_PRESS, sigma_P=SIGMA_P,
        exclude_press_sensors=['TC1'],
    )
    return J, details


def eval_stage_B(params_dict):
    """Run all tests with frozen Stage A + this Stage B, return (J_B, details)."""
    temp_rmse_per_test = {}
    for tid in TEST_IDS:
        ch = run_sim_calibrated(tid, stage_a=_frozen_a, stage_b=params_dict)
        res = extract_results(tid, ch)
        temp_rmse_per_test[tid] = res['temp_rmse']

    temp_sensors_per_test = {tid: TEMP_SENSORS[exp_data[tid]['chimney_type']]
                             for tid in TEST_IDS}

    J, details = compute_stage_B_objective(
        temp_rmse_per_test=temp_rmse_per_test,
        test_ids=TEST_IDS,
        temp_sensors_per_test=temp_sensors_per_test,
    )
    return J, details


def eval_stage_C(params_dict):
    """Run all tests with frozen A+B + this Stage C, return (J_C, details)."""
    ss_mdot_sim = {}
    ss_press_top_sim = {}
    for tid in TEST_IDS:
        ch = run_sim_calibrated(tid, stage_a=_frozen_a, stage_b=_frozen_b,
                                stage_c=params_dict)
        res = extract_results(tid, ch)
        ss_mdot_sim[tid] = res['mdot_ss']
        ss_press_top_sim[tid] = res['press_ss'].get('TOP', np.nan)

    exp_press_top = {tid: exp_ss_press[tid].get('TOP', np.nan) for tid in TEST_IDS}

    J, details = compute_stage_C_objective(
        ss_mdot_sim=ss_mdot_sim,
        ss_mdot_exp=exp_ss_mdot,
        ss_mdot_exp_std=exp_ss_mdot_std,
        ss_press_top_sim=ss_press_top_sim,
        ss_press_top_exp=exp_press_top,
        test_ids=TEST_IDS,
    )
    return J, details


print('Evaluation functions defined.')

## Baseline evaluation

In [ ]:
print('Running baseline (paper/results defaults)...')
t0 = _time.time()
J_A_baseline, details_A_baseline = eval_stage_A(BASELINE_A.to_dict())
J_B_baseline, details_B_baseline = eval_stage_B(BASELINE_B.to_dict())
print(f'Baseline: J_A={J_A_baseline:.4f}  J_B={J_B_baseline:.4f}  ({_time.time()-t0:.0f}s)')
print(f'  J_A breakdown: J_mdot={details_A_baseline["J_mdot"]:.3f}, '
      f'J_press={details_A_baseline["J_press"]:.3f}')
for tid in TEST_IDS:
    d = details_A_baseline['per_test'].get(tid, {})
    print(f'  {tid}: mdot_sim={d.get("mdot_sim",0):.1f} vs exp={d.get("mdot_exp",0):.1f} g/s')

## Stage A: Momentum / Resistance Calibration

In [ ]:
print(f'=== Stage A: Coarse sweep ({N_COARSE_A} samples) ===')
cal_A = StagedCalibrator(OUT_ROOT / 'stage_A', seed=RNG_SEED)

t0 = _time.time()
results_A = cal_A.coarse_sweep(StageAParams, N_COARSE_A, eval_stage_A)
print(f'Coarse sweep done in {_time.time()-t0:.0f}s')

top_A = cal_A.get_top_n(results_A, n=10)
print(f'\nTop {len(top_A)} results:')
for i, r in enumerate(top_A):
    p = r['params']
    print(f'  #{i+1}: J={r["objective"]:.4f}  '
          f'A_leak={p["A_leak_cm2"]:.1f}  Cd={p["Cd"]:.3f}  '
          f'K_stove={p["K_stove"]:.2f}  K_mult={p["K_minor_mult"]:.2f}')

In [ ]:
print(f'=== Stage A: Local refinement (top {N_REFINE_TOP} candidates) ===')
best_A_overall = {'objective': 1e12}

for i, candidate in enumerate(top_A[:N_REFINE_TOP]):
    print(f'\n--- Refining candidate #{i+1} (J={candidate["objective"]:.4f}) ---')
    x0 = StageAParams(**{k: v for k, v in candidate['params'].items()
                         if k != 'BOUNDS'}).to_array()
    refined = cal_A.local_refine(
        StageAParams, x0, eval_stage_A,
        method='Nelder-Mead', maxiter=REFINE_MAXITER,
    )
    if refined['objective'] < best_A_overall.get('objective', 1e12):
        best_A_overall = refined

print(f'\n=== Stage A BEST: J={best_A_overall["objective"]:.4f} ===')
print(f'  Params: {best_A_overall["params"]}')
print(f'  Baseline was: J={J_A_baseline:.4f}')
improvement = (1 - best_A_overall['objective'] / max(J_A_baseline, 1e-12)) * 100
print(f'  Improvement: {improvement:.1f}%')

# Freeze Stage A params
_frozen_a = best_A_overall['params']
cal_A.save_best_params(_frozen_a, 'best_params.json')
cal_A.save_results(results_A, 'coarse_results.json')

In [ ]:
# Stage A summary plots
valid_A = [r for r in results_A if r.get('valid', False)]
if len(valid_A) > 1:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, name in zip(axes, StageAParams.names()):
        xs = [r['params'][name] for r in valid_A]
        ys = [r['objective'] for r in valid_A]
        ax.scatter(xs, ys, s=20, alpha=0.7)
        ax.set_xlabel(name)
        ax.set_ylabel('J_A')
        ax.grid(True, alpha=0.3)
        # Mark best
        bx = best_A_overall['params'][name]
        by = best_A_overall['objective']
        ax.scatter([bx], [by], s=80, color='red', zorder=5, marker='*')
    fig.suptitle('Stage A: Objective vs. knobs', fontsize=12)
    savefig(fig, OUT_ROOT / 'stage_A' / 'fig_sweep_scatter.png')
    plt.show()
    plt.close(fig)

# Before/after mdot comparison
print('\nRunning before/after comparison for Stage A...')
mdot_before = {}
mdot_after = {}
for tid in TEST_IDS:
    ch_b = run_sim_calibrated(tid, stage_a=BASELINE_A)
    ch_a = run_sim_calibrated(tid, stage_a=_frozen_a)
    rb = extract_results(tid, ch_b)
    ra = extract_results(tid, ch_a)
    mdot_before[tid] = rb['mdot_ss']
    mdot_after[tid]  = ra['mdot_ss']

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TEST_IDS))
w = 0.25
ax.bar(x - w, [mdot_before[t] for t in TEST_IDS], w, label='Before (baseline)', color='C0', alpha=0.7)
ax.bar(x,     [mdot_after[t] for t in TEST_IDS],  w, label='After (Stage A)',    color='C2', alpha=0.7)
ax.bar(x + w, [exp_ss_mdot[t] for t in TEST_IDS], w,
       yerr=[exp_ss_mdot_std[t] for t in TEST_IDS],
       label='Exp pitot', color='C1', alpha=0.7, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels([f'{t}\n({exp_data[t]["chimney_type"]})' for t in TEST_IDS])
ax.set_ylabel('SS mdot (g/s)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_title('Stage A: mdot before/after calibration')
savefig(fig, OUT_ROOT / 'stage_A' / 'fig_mdot_before_after.png')
plt.show()
plt.close(fig)

## Stage B: Heat-Loss Calibration

In [ ]:
print(f'=== Stage B: Coarse sweep ({N_COARSE_B} samples) ===')
print(f'  Using frozen Stage A: {_frozen_a}')
cal_B = StagedCalibrator(OUT_ROOT / 'stage_B', seed=RNG_SEED + 1)

t0 = _time.time()
results_B = cal_B.coarse_sweep(StageBParams, N_COARSE_B, eval_stage_B)
print(f'Coarse sweep done in {_time.time()-t0:.0f}s')

top_B = cal_B.get_top_n(results_B, n=10)
print(f'\nTop {len(top_B)} results:')
for i, r in enumerate(top_B):
    p = r['params']
    print(f'  #{i+1}: J={r["objective"]:.4f}  '
          f'h_in={p["h_inner_mult"]:.3f}  h_out={p["h_outer_mult"]:.3f}  '
          f'eps_s={p["eps_inner_scale"]:.3f}')

In [ ]:
print(f'=== Stage B: Local refinement (top {N_REFINE_TOP} candidates) ===')
best_B_overall = {'objective': 1e12}

for i, candidate in enumerate(top_B[:N_REFINE_TOP]):
    print(f'\n--- Refining candidate #{i+1} (J={candidate["objective"]:.4f}) ---')
    x0 = StageBParams(**{k: v for k, v in candidate['params'].items()
                         if k != 'BOUNDS'}).to_array()
    refined = cal_B.local_refine(
        StageBParams, x0, eval_stage_B,
        method='Nelder-Mead', maxiter=REFINE_MAXITER,
    )
    if refined['objective'] < best_B_overall.get('objective', 1e12):
        best_B_overall = refined

print(f'\n=== Stage B BEST: J={best_B_overall["objective"]:.4f} ===')
print(f'  Params: {best_B_overall["params"]}')
print(f'  Baseline was: J={J_B_baseline:.4f}')
improvement_B = (1 - best_B_overall['objective'] / max(J_B_baseline, 1e-12)) * 100
print(f'  Improvement: {improvement_B:.1f}%')

# Freeze Stage B params
_frozen_b = best_B_overall['params']
cal_B.save_best_params(_frozen_b, 'best_params.json')
cal_B.save_results(results_B, 'coarse_results.json')

In [ ]:
# Stage B summary plots
valid_B = [r for r in results_B if r.get('valid', False)]
if len(valid_B) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, name in zip(axes, StageBParams.names()):
        xs = [r['params'][name] for r in valid_B]
        ys = [r['objective'] for r in valid_B]
        ax.scatter(xs, ys, s=20, alpha=0.7)
        ax.set_xlabel(name)
        ax.set_ylabel('J_B')
        ax.grid(True, alpha=0.3)
        bx = best_B_overall['params'][name]
        by = best_B_overall['objective']
        ax.scatter([bx], [by], s=80, color='red', zorder=5, marker='*')
    fig.suptitle('Stage B: Objective vs. knobs', fontsize=12)
    savefig(fig, OUT_ROOT / 'stage_B' / 'fig_sweep_scatter.png')
    plt.show()
    plt.close(fig)

# Before/after temperature comparison for representative test
print(f'\nTemperature before/after for {REP_TEST}...')
ch_before = run_sim_calibrated(REP_TEST, stage_a=_frozen_a, stage_b=BASELINE_B)
ch_after  = run_sim_calibrated(REP_TEST, stage_a=_frozen_a, stage_b=_frozen_b)

ct = exp_data[REP_TEST]['chimney_type']
sensors = TEMP_SENSORS[ct]
fig, axes = plt.subplots(len(sensors), 1, figsize=(10, 3 * len(sensors)), sharex=True)
if len(sensors) == 1:
    axes = [axes]
t_sim = np.arange(ch_after.T_gas.shape[0]) * ch_after.dt

for ax, sensor in zip(axes, sensors):
    td = exp_data[REP_TEST]['temperature'][sensor]
    t_exp = np.asarray(td['time_s'], dtype=float)
    T_exp = np.asarray(td['temperature_C'], dtype=float)

    seg = find_segment(ch_after, *SENSOR_MAP[ct][sensor])
    T_before = ch_before.T_gas[:, seg] - 273.15
    T_after  = ch_after.T_gas[:, seg] - 273.15

    ax.plot(t_exp / 60, T_exp, 'k-', lw=0.8, alpha=0.7, label='exp')
    ax.plot(t_sim / 60, T_before, 'C0--', lw=1.2, label='before (B baseline)')
    ax.plot(t_sim / 60, T_after,  'C2--', lw=1.2, label='after (B calibrated)')
    ax.set_ylabel(f'{sensor} T (C)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (min)')
axes[0].set_title(f'Stage B: Temperature before/after ({REP_TEST})')
savefig(fig, OUT_ROOT / 'stage_B' / 'fig_T_before_after.png')
plt.show()
plt.close(fig)

## Stage C: Cap Cp (optional)

Run only if cross-test mdot bias varies systematically with wind speed.

In [ ]:
# Stage C is optional. Uncomment to run.
# To run: set N_COARSE_C and uncomment the block below.

RUN_STAGE_C = False
best_C = None

if RUN_STAGE_C:
    N_COARSE_C = 15
    print(f'=== Stage C: Coarse sweep ({N_COARSE_C} samples) ===')
    cal_C = StagedCalibrator(OUT_ROOT / 'stage_C', seed=RNG_SEED + 2)
    results_C = cal_C.coarse_sweep(StageCParams, N_COARSE_C, eval_stage_C)
    top_C = cal_C.get_top_n(results_C, n=5)
    for i, r in enumerate(top_C):
        print(f'  #{i+1}: J={r["objective"]:.4f}  Cp={r["params"]["Cp_override"]:.3f}')

    # Refine top 1
    x0 = StageCParams(**top_C[0]['params']).to_array()
    best_C = cal_C.local_refine(StageCParams, x0, eval_stage_C, maxiter=20)
    _frozen_c = best_C['params']
    cal_C.save_best_params(_frozen_c, 'best_params.json')
    print(f'Stage C BEST: Cp={_frozen_c["Cp_override"]:.3f}  J={best_C["objective"]:.4f}')
else:
    print('Stage C skipped (RUN_STAGE_C=False). Using default Cp=-0.30.')

In [ ]:
# Stage D: Cracked-door refinement (optional)
# This stage is a stub -- requires cracked-door objective and pitot data
# in the early transient phase, which is noisy.

print('Stage D skipped (placeholder for cracked-door refinement).')
print('  Knobs: door_area_scale, door_ramp_s')
print('  Would optimize against cracked-door mdot + P + T.')

## Final calibrated results

In [ ]:
print('=== Running final calibrated simulations ===')
print(f'  Stage A: {_frozen_a}')
print(f'  Stage B: {_frozen_b}')

final_results = {}
t0 = _time.time()
for tid in TEST_IDS:
    ch = run_sim_calibrated(tid, stage_a=_frozen_a, stage_b=_frozen_b,
                            stage_c=_frozen_c)
    final_results[tid] = {'ch': ch, 'extracted': extract_results(tid, ch)}
    print(f'  {tid}: mdot_ss={final_results[tid]["extracted"]["mdot_ss"]:.1f} g/s')

print(f'Done in {_time.time()-t0:.0f}s.')

In [ ]:
# Final T + P overlays for representative test
final_dir = OUT_ROOT / 'final_results_plots'

for rep_test in [REP_TEST]:
    ct = exp_data[rep_test]['chimney_type']
    ch = final_results[rep_test]['ch']
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt
    t_close = exp_data[rep_test]['config']['door_events']['offsets_s']['door_closed']

    # Temperature
    sensors_t = TEMP_SENSORS[ct]
    fig, axes = plt.subplots(len(sensors_t), 1, figsize=(10, 3 * len(sensors_t)),
                             sharex=True)
    if len(sensors_t) == 1: axes = [axes]
    for ax, sensor in zip(axes, sensors_t):
        td = exp_data[rep_test]['temperature'][sensor]
        ax.plot(np.asarray(td['time_s'])/60, np.asarray(td['temperature_C']),
                'k-', lw=0.8, alpha=0.7, label='exp')
        seg = find_segment(ch, *SENSOR_MAP[ct][sensor])
        ax.plot(t_sim/60, ch.T_gas[:, seg]-273.15, 'C2--', lw=1.3, label='sim (cal)')
        ax.axvline(t_close/60, color='gray', ls=':', alpha=0.5)
        ax.set_ylabel(f'{sensor} T (C)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Final calibrated: {rep_test} ({ct}) -- Temperature')
    savefig(fig, final_dir / f'{rep_test}_T_overlay.png')
    plt.show()
    plt.close(fig)

    # Pressure
    sensors_p = PRESS_SENSORS[ct]
    seg_map = {s: find_segment(ch, *SENSOR_MAP[ct][s]) for s in sensors_p}
    n = ch.T_gas.shape[0]
    step_indices = np.arange(0, n, PRESSURE_SUBSAMPLE)
    p_times = step_indices * ch.dt
    unique_segs = list(set(seg_map.values()))
    seg_P = {seg: np.zeros(len(step_indices)) for seg in unique_segs}
    for i, step in enumerate(step_indices):
        prof = ch.get_static_pressure_profile(timestep=int(step))
        for seg in unique_segs:
            seg_P[seg][i] = prof['P_gauge_Pa'][seg]

    fig, axes = plt.subplots(len(sensors_p), 1, figsize=(10, 2.8 * len(sensors_p)),
                             sharex=True)
    if len(sensors_p) == 1: axes = [axes]
    for ax, sensor in zip(axes, sensors_p):
        pd_ = exp_data[rep_test]['pressure'][sensor]
        ax.plot(np.asarray(pd_['time_s'])/60, np.asarray(pd_['pressure_Pa']),
                'k-', lw=0.8, alpha=0.7, label='exp')
        ax.plot(p_times/60, seg_P[seg_map[sensor]], 'C2--', lw=1.3, label='sim (cal)')
        ax.axhline(0, color='gray', ls=':', alpha=0.4)
        ax.axvline(t_close/60, color='gray', ls=':', alpha=0.5)
        ax.set_ylabel(f'{sensor} P (Pa)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Time (min)')
    axes[0].set_title(f'Final calibrated: {rep_test} ({ct}) -- Pressure')
    savefig(fig, final_dir / f'{rep_test}_P_overlay.png')
    plt.show()
    plt.close(fig)

# mdot SS comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TEST_IDS))
w = 0.3
ax.bar(x - w/2, [final_results[t]['extracted']['mdot_ss'] for t in TEST_IDS],
       w, label='Sim (calibrated)', color='C2', alpha=0.85)
ax.bar(x + w/2, [exp_ss_mdot[t] for t in TEST_IDS], w,
       yerr=[exp_ss_mdot_std[t] for t in TEST_IDS],
       label='Exp pitot', color='C1', alpha=0.85, capsize=4)
for i, tid in enumerate(TEST_IDS):
    r = final_results[tid]['extracted']['mdot_ss'] / exp_ss_mdot[tid]
    y_top = max(final_results[tid]['extracted']['mdot_ss'],
                exp_ss_mdot[tid] + exp_ss_mdot_std[tid]) + 0.5
    ax.text(i, y_top, f'{r:.2f}x', ha='center', fontsize=9, color='dimgray')
ax.set_xticks(x)
ax.set_xticklabels([f'{t}\n({exp_data[t]["chimney_type"]})' for t in TEST_IDS])
ax.set_ylabel('SS mdot (g/s)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_title('Final calibrated: SS mass flow comparison')
savefig(fig, final_dir / 'fig_mdot_ss_final.png')
plt.show()
plt.close(fig)

## Report

In [ ]:
# Build RMSE tables for final calibrated results
rmse_T_rows = []
rmse_P_rows = []
for tid in TEST_IDS:
    ct = exp_data[tid]['chimney_type']
    ch = final_results[tid]['ch']
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt
    _, _, t_cls0, t_cls1 = get_phase_windows(tid)

    seg_map = {s: find_segment(ch, *SENSOR_MAP[ct][s]) for s in PRESS_SENSORS[ct]}
    n = ch.T_gas.shape[0]
    step_indices = np.arange(0, n, PRESSURE_SUBSAMPLE)
    p_times = step_indices * ch.dt
    unique_segs = list(set(seg_map.values()))
    seg_P = {seg: np.zeros(len(step_indices)) for seg in unique_segs}
    for i, step in enumerate(step_indices):
        prof = ch.get_static_pressure_profile(timestep=int(step))
        for seg in unique_segs:
            seg_P[seg][i] = prof['P_gauge_Pa'][seg]

    for sensor in TEMP_SENSORS[ct]:
        seg = find_segment(ch, *SENSOR_MAP[ct][sensor])
        sim_T = ch.T_gas[:, seg] - 273.15
        td = exp_data[tid]['temperature'][sensor]
        et = np.asarray(td['time_s'], dtype=float)
        ev = np.asarray(td['temperature_C'], dtype=float)
        rmse, bias = compute_rmse_bias(et, ev, t_sim, sim_T, (t_cls0, t_cls1))
        rmse_T_rows.append({'Test': tid, 'Sensor': sensor,
                            'RMSE_C': rmse, 'Bias_C': bias})

    for sensor in PRESS_SENSORS[ct]:
        pd_ = exp_data[tid]['pressure'][sensor]
        et = np.asarray(pd_['time_s'], dtype=float)
        ev = np.asarray(pd_['pressure_Pa'], dtype=float)
        rmse, bias = compute_rmse_bias(et, ev, p_times, seg_P[seg_map[sensor]],
                                       (t_cls0, t_cls1))
        rmse_P_rows.append({'Test': tid, 'Sensor': sensor,
                            'RMSE_Pa': rmse, 'Bias_Pa': bias})

df_T = pd.DataFrame(rmse_T_rows)
df_P = pd.DataFrame(rmse_P_rows)

# mdot table
mdot_rows = []
for tid in TEST_IDS:
    m_s = final_results[tid]['extracted']['mdot_ss']
    m_e = exp_ss_mdot[tid]
    mdot_rows.append({'Test': tid, 'Type': exp_data[tid]['chimney_type'],
                      'mdot_sim': round(m_s, 1), 'mdot_exp': round(m_e, 1),
                      'ratio': round(m_s / m_e, 2) if m_e > 0 else np.nan})
df_mdot = pd.DataFrame(mdot_rows)

# Save combined best params
combined_params = {
    'stage_A': _frozen_a,
    'stage_B': _frozen_b,
    'stage_C': _frozen_c,
    'baseline_A': BASELINE_A.to_dict(),
    'baseline_B': BASELINE_B.to_dict(),
    'J_A_baseline': J_A_baseline,
    'J_A_best': best_A_overall['objective'],
    'J_B_baseline': J_B_baseline,
    'J_B_best': best_B_overall['objective'],
}
with open(OUT_ROOT / 'best_params_combined.json', 'w') as f:
    json.dump(combined_params, f, indent=2, default=str)

report = f"""
-----BEGIN REPORT-----
# Staged Calibration Report
Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
Branch: paper/results
Tests: {', '.join(TEST_IDS)}
Solver: relax_to_steady, no distributed leakage, no BE-CV

## Stage A: Momentum / Resistance (closed-door)
Baseline: J_A = {J_A_baseline:.4f}
Best:     J_A = {best_A_overall['objective']:.4f}
Params:
  A_leak_cm2   = {_frozen_a['A_leak_cm2']:.2f}
  Cd           = {_frozen_a['Cd']:.4f}
  K_stove      = {_frozen_a['K_stove']:.4f}
  K_minor_mult = {_frozen_a['K_minor_mult']:.4f}
Samples: {N_COARSE_A} coarse + {N_REFINE_TOP} refined

## Stage B: Heat-Loss (closed-door)
Baseline: J_B = {J_B_baseline:.4f}
Best:     J_B = {best_B_overall['objective']:.4f}
Params:
  h_inner_mult   = {_frozen_b['h_inner_mult']:.4f}
  h_outer_mult   = {_frozen_b['h_outer_mult']:.4f}
  eps_inner_scale = {_frozen_b['eps_inner_scale']:.4f}
Samples: {N_COARSE_B} coarse + {N_REFINE_TOP} refined

## Stage C: Cap Cp
{'Skipped (RUN_STAGE_C=False)' if not RUN_STAGE_C else f'Cp_override = {_frozen_c["Cp_override"]:.3f}'}

## Stage D: Cracked-door
Skipped (placeholder)

## Temperature RMSE/Bias (closed-door) [C]
{df_T.to_markdown(index=False, floatfmt='.1f')}

## Pressure RMSE/Bias (closed-door) [Pa]
{df_P.to_markdown(index=False, floatfmt='.2f')}

## Steady-state mass flow
{df_mdot.to_markdown(index=False)}

## Output directory
{OUT_ROOT}
-----END REPORT-----
"""
print(report)